# Análise de Dados - E-commerce Olist

## Tratamento e preparação dos dados

In [15]:
import pandas as pd 
import numpy as np

## Carregamento dos dados

Os datasets serão carregados novamente a partir dos arquivos brutos (`raw`). Nesta etapa, os dados ainda não sofrerão alterações e serão utilizados como base para o processo de tratamento.

In [16]:
df_clientes = pd.read_csv("../data/raw/olist_customers_dataset.csv")

df_pedidos = pd.read_csv("../data/raw/olist_orders_dataset.csv")

df_itens = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

df_pagamentos = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

df_avaliacoes = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

df_produtos = pd.read_csv("../data/raw/olist_products_dataset.csv")

df_categorias = pd.read_csv("../data/raw/product_category_name_translation.csv")

## Conversão das colunas de data

As colunas que representam datas estão armazenadas inicialmente como `str`. Nesta etapa, essas colunas serão convertidas para o tipo `datetime`, permitindo a realização de cálculos e análises temporais nas etapas posteriores.

### Conversão das datas de pedidos

As colunas relacionadas às datas dos pedidos serão convertidas para o tipo `datetime`, permitindo análises temporais e cálculos relacionados aos prazos de entrega.

In [17]:
colunas_data_pedidos = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for coluna in colunas_data_pedidos:
    try:
        pd.to_datetime(df_pedidos[coluna])
        print(f"{coluna}: OK")
    except Exception as erro:
        print(f"{coluna}: ERRO")
        print(erro)

order_purchase_timestamp: OK
order_approved_at: OK
order_delivered_carrier_date: OK
order_delivered_customer_date: OK
order_estimated_delivery_date: OK


Como todas as colunas foram validadas com sucesso, elas serão convertidas para o tipo `datetime`.

In [18]:
for coluna in colunas_data_pedidos:
    df_pedidos[coluna] = pd.to_datetime(df_pedidos[coluna])

In [19]:
df_pedidos.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

### Conversão da data dos itens

A coluna `shipping_limit_date` representa a data limite de envio dos itens e está armazenada inicialmente como `str`. A coluna será validada e posteriormente convertida para o tipo `datetime`.

In [20]:
df_itens["shipping_limit_date"] = pd.to_datetime(df_itens["shipping_limit_date"])

In [21]:
df_itens.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

### Conversão das datas das avaliações

As colunas `review_creation_date` e `review_answer_timestamp` representam datas relacionadas às avaliações dos pedidos e estão armazenadas inicialmente como `str`. As colunas serão convertidas para o tipo `datetime`.

In [22]:
colunas_data_avaliacoes = [
    "review_creation_date",
    "review_answer_timestamp"
]

for coluna in colunas_data_avaliacoes:
    try:
        pd.to_datetime(df_avaliacoes[coluna])
        print(f"{coluna}: OK")
    except Exception as erro:
        print(f"{coluna}: ERRO")
        print(erro)

review_creation_date: OK
review_answer_timestamp: OK


As duas colunas foram validadas com sucesso, elas serão convertidas para o tipo `datetime`.

In [23]:
for coluna in colunas_data_avaliacoes:
    df_avaliacoes[coluna] = pd.to_datetime(df_avaliacoes[coluna])

In [24]:
df_avaliacoes.dtypes

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

## Tratamento dos valores ausentes

Os valores ausentes identificados durante a exploração inicial serão analisados individualmente antes da aplicação de qualquer tratamento. A estratégia adotada dependerá do significado da ausência e da importância da variável para as análises posteriores.

In [25]:
tabelas = {
    "Clientes": df_clientes,
    "Pedidos": df_pedidos,
    "Itens": df_itens,
    "Pagamentos": df_pagamentos,
    "Avaliações": df_avaliacoes,
    "Produtos": df_produtos,
    "Categorias": df_categorias
}

for nome, df in tabelas.items():
    print(f"{nome}")
    print(df.isna().sum())
    print("-" * 30)
    

Clientes
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
------------------------------
Pedidos
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
------------------------------
Itens
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
------------------------------
Pagamentos
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
------------------------------
Avaliações
review_id           

### Análise dos valores ausentes em pedidos

Os valores ausentes nas colunas relacionadas à aprovação e entrega dos pedidos serão analisados em conjunto com o `order_status`, a fim de verificar se as ausências estão associadas à situação de cada pedido antes de definir uma estratégia de tratamento.

In [29]:
df_pedidos.loc[df_pedidos["order_approved_at"].isna(), "order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [31]:
df_pedidos.loc[df_pedidos["order_delivered_carrier_date"].isna(), "order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [32]:
df_pedidos.loc[df_pedidos["order_delivered_customer_date"].isna(), "order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

#### Decisão de tratamento - Pedidos

Os valores ausentes nas colunas de aprovação e entrega foram analisados em conjunto com o status dos pedidos.

A maior parte das ausências está associada a pedidos que não concluíram determinadas etapas do processo, como pedidos `canceled`, `unavailable`, `processing`, `invoiced` ou `shipped`. Nesses casos, a ausência da respectiva data é compatível com o status do pedido.

Também foram identificados alguns pedidos com status `delivered` que apresentam datas ausentes, indicando registros incompletos na base.

Como não é possível determinar com segurança as datas ausentes, os valores serão mantidos como `NaT`, evitando a criação de informações artificiais ou a exclusão desnecessária de pedidos.

### Análise dos valores ausentes em avaliações

O dataset de avaliações apresenta valores ausentes nas colunas `review_comment_title` e `review_comment_message`. Entretanto, a coluna `review_score` não apresenta valores ausentes, indicando que os registros possuem uma nota de avaliação mesmo quando não há um comentário textual.

Como o preenchimento de título e comentário é opcional e não é possível determinar o conteúdo que seria escrito pelos clientes, os valores ausentes serão mantidos. Essa decisão preserva as avaliações e evita a criação artificial de informações.

### Análise dos valores ausentes em produtos

Os valores ausentes encontrados no dataset de produtos serão investigados antes da definição da estratégia de tratamento, buscando identificar se as ausências estão concentradas nos mesmos registros.

In [33]:
colunas_nulas_produtos = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

df_produtos[colunas_nulas_produtos].isna().all(axis=1).sum()

np.int64(610)

In [34]:
colunas_dimensoes_produtos = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

df_produtos[colunas_dimensoes_produtos].isna().all(axis=1).sum()

np.int64(2)

In [35]:
df_produtos.loc[
    df_produtos["product_weight_g"].isna(),
    [
        "product_id",
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
produtos_sem_categoria = df_produtos.loc[df_produtos["product_category_name"].isna(), "product_id"]

df_itens["product_id"].isin(produtos_sem_categoria).sum()

np.int64(1603)

In [38]:
df_itens.loc[df_itens["product_id"].isin(produtos_sem_categoria), "product_id"].nunique()

610

#### Decisão de tratamento - Produtos

Os 610 produtos sem categoria foram mantidos, pois todos possuem registros associados na tabela de itens dos pedidos, totalizando 1.603 ocorrências.

Para preservar esses registros nas futuras análises por categoria, os valores ausentes de `product_category_name` serão identificados como `sem_categoria`.

Os valores ausentes relacionados ao tamanho do nome, descrição, quantidade de fotos, peso e dimensões serão mantidos como `NaN`, pois não é possível determinar seus valores reais com segurança e seu preenchimento poderia introduzir informações artificiais nos dados.

In [39]:
df_produtos["product_category_name"] = (df_produtos["product_category_name"].fillna("sem_categoria"))

In [40]:
df_produtos["product_category_name"].isna().sum()

np.int64(0)